# Food Waste Analysis — Results in the Notebook

This notebook executes the analysis blocks from **`food_waste_analysis.sql`** and displays every query result below its title.

## Before you run
1. Keep `food_waste_analysis.sql` at `db/queries/food_waste_analysis.sql` relative to the project root.
2. Make sure you have already created and loaded the source tables from `food_waste_queries.sql` in the PostgreSQL database:
   `country`, `commodity`, `method`, `activity`, `source`, `observation`, and `observation_activity`.
3. Update the database connection settings in the next cell.
4. Run the notebook from top to bottom.

The notebook runs the `setup` block first (creates lookup tables and views), then runs every named analytical query block and displays its result inline. Results are also kept in the `results` dictionary for further notebook analysis.

**Important:** This notebook does not create or load the original source tables. It expects them to exist already.

In [6]:
# If needed, install the required packages by running this cell once:
%pip install pandas psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path
import re
import pandas as pd
import psycopg2
from getpass import getpass
from IPython.display import display

# SQL file is stored in db/queries relative to the project root.
SQL_FILE = Path("../db/queries/food_waste_analysis.sql")

# PostgreSQL connection settings — edit these values.
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "food_waste_db",
    "user": "airflow",
    "password": 'airflow',  # You will be prompted securely below.
}

if DB_CONFIG["password"] is None:
    DB_CONFIG["password"] = getpass("PostgreSQL password: ")

if not SQL_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {SQL_FILE.resolve()}\n"
        "Make sure db/queries/food_waste_analysis.sql exists in the project."
    )

print("SQL file:", SQL_FILE.resolve())

SQL file: D:\My_projects\data-engineer\waste-food-project\db\queries\food_waste_analysis.sql
CSV output folder: D:\My_projects\data-engineer\waste-food-project\notebooks\food_waste_analysis_results


## 1. Read and split the SQL file

The notebook uses the metadata comments in your SQL file (`-- name:`, `-- page:`, `-- title:`, `-- question:`, and `-- visual:`) to identify the setup block and each analytical query. Keep those comments in the SQL file.

In [8]:
sql_text = SQL_FILE.read_text(encoding="utf-8")

# Each block starts at a line containing: -- name: <block_name>
block_pattern = re.compile(
    r"(?m)^--\s*name:\s*(.*?)\s*$"
)

matches = list(block_pattern.finditer(sql_text))
if not matches:
    raise ValueError("No '-- name:' block markers were found in the SQL file.")

blocks = []
for i, match in enumerate(matches):
    start = match.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(sql_text)
    block_text = sql_text[start:end].strip()

    def get_meta(field):
        m = re.search(rf"(?m)^--\s*{re.escape(field)}:\s*(.*?)\s*$", block_text)
        return m.group(1).strip() if m else ""

    # Remove the metadata comment lines from the SQL sent to PostgreSQL.
    query_sql = re.sub(
        r"(?m)^--\s*(?:name|page|title|question|visual):.*(?:\n|$)",
        "",
        block_text
    ).strip()

    blocks.append({
        "name": get_meta("name"),
        "page": get_meta("page"),
        "title": get_meta("title"),
        "question": get_meta("question"),
        "visual": get_meta("visual"),
        "sql": query_sql,
    })

print(f"Found {len(blocks)} SQL blocks.")
display(pd.DataFrame([{k: b[k] for k in ["name", "page", "title", "visual"]} for b in blocks]))

Found 33 SQL blocks.


,name,page,title,visual
0,setup,0 Setup,Lookup tables and analysis views,none
1,kpis,1 Overview,Headline numbers,KPI cards
2,loss_distribution,1 Overview,Distribution of recorded loss,Column chart
3,yearly_coverage,1 Overview,Records and countries per year,Column chart + line (countries)
4,method_summary,1 Overview,Data collection methods,Bar chart
5,top_crops,2 Crops,Crops with the highest median loss,Horizontal bar chart
6,lowest_loss_crops,2 Crops,Crops with the lowest median loss,Horizontal bar chart
7,top_crops_measured,2 Crops,Crop ranking using measured data only,Bar chart with rank shift
8,product_group_summary,2 Crops,Loss by product group,Bar chart
9,crop_variability,2 Crops,Crops with the most uncertain loss,Range bars (p10 to p90)


## 2. Connect to PostgreSQL

In [9]:
connection = psycopg2.connect(**DB_CONFIG)
connection.autocommit = False

with connection.cursor() as cursor:
    cursor.execute("SELECT current_database(), current_user, version();")
    db_name, db_user, db_version = cursor.fetchone()

print(f"Connected successfully to database: {db_name}")
print(f"Connected user: {db_user}")
print(db_version.splitlines()[0])

Connected successfully to database: food_waste_db
Connected user: airflow
PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Run setup

The `setup` block drops and recreates the analysis lookup tables and views. It is designed to be re-runnable, but it replaces the existing analysis views/tables with the definitions in your SQL file.

If setup fails, the notebook rolls back the transaction and shows the PostgreSQL error.

In [10]:
setup_blocks = [b for b in blocks if b["name"] == "setup"]
if len(setup_blocks) != 1:
    raise ValueError(f"Expected exactly one setup block; found {len(setup_blocks)}.")

try:
    with connection.cursor() as cursor:
        cursor.execute(setup_blocks[0]["sql"])
    connection.commit()
    print("Setup completed: lookup tables and analysis/dashboard views are ready.")
except Exception:
    connection.rollback()
    raise

Setup completed: lookup tables and analysis/dashboard views are ready.


## 4. Execute all analytical queries and display their results

Each query is executed separately. Its title and complete result table appear inline below. A small execution summary is displayed after all queries finish.

If any query fails, the transaction is rolled back and the error identifies the failing block.

In [11]:
query_blocks = [b for b in blocks if b["name"] != "setup"]
results = {}
execution_log = []

for block in query_blocks:
    block_name = block["name"]
    if not block_name:
        continue

    try:
        df = pd.read_sql_query(block["sql"], connection)
        results[block_name] = df
        print(f"\n{block['page']} | {block['title'] or block_name} ({block_name})")
        if block["question"]:
            print(block["question"])
        display(df)
        execution_log.append({
            "name": block_name,
            "page": block["page"],
            "title": block["title"],
            "rows": len(df),
            "columns": len(df.columns),
            "status": "Success",
        })
        print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
    except Exception as error:
        connection.rollback()
        execution_log.append({
            "name": block_name,
            "page": block["page"],
            "title": block["title"],
            "rows": None,
            "columns": None,
            "status": f"Failed: {error}",
        })
        print(f"✗ Failed block: {block_name}")
        raise

execution_summary = pd.DataFrame(execution_log)
display(execution_summary)

C:\Users\GM\AppData\Local\Temp\ipykernel_14280\722800803.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(block["sql"], connection)


✓ kpis: 1 rows → food_waste_analysis_results\kpis.csv
✓ loss_distribution: 6 rows → food_waste_analysis_results\loss_distribution.csv


C:\Users\GM\AppData\Local\Temp\ipykernel_14280\722800803.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(block["sql"], connection)
C:\Users\GM\AppData\Local\Temp\ipykernel_14280\722800803.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(block["sql"], connection)


✓ yearly_coverage: 27 rows → food_waste_analysis_results\yearly_coverage.csv


C:\Users\GM\AppData\Local\Temp\ipykernel_14280\722800803.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(block["sql"], connection)


✓ method_summary: 12 rows → food_waste_analysis_results\method_summary.csv


... (49 outputs left out)

✓ outlier_records: 20 rows → food_waste_analysis_results\outlier_records.csv
✓ data_quality_checks: 6 rows → food_waste_analysis_results\data_quality_checks.csv


C:\Users\GM\AppData\Local\Temp\ipykernel_14280\722800803.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(block["sql"], connection)
C:\Users\GM\AppData\Local\Temp\ipykernel_14280\722800803.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(block["sql"], connection)


✓ check_product_groups: 15 rows → food_waste_analysis_results\check_product_groups.csv


,name,page,title,rows,columns,csv_file,status
0,kpis,1 Overview,Headline numbers,1,11,food_waste_analysis_results\kpis.csv,Success
1,loss_distribution,1 Overview,Distribution of recorded loss,6,3,food_waste_analysis_results\loss_distribution.csv,Success
2,yearly_coverage,1 Overview,Records and countries per year,27,9,food_waste_analysis_results\yearly_coverage.csv,Success
3,method_summary,1 Overview,Data collection methods,12,6,food_waste_analysis_results\method_summary.csv,Success
4,top_crops,2 Crops,Crops with the highest median loss,20,8,food_waste_analysis_results\top_crops.csv,Success
5,lowest_loss_crops,2 Crops,Crops with the lowest median loss,15,6,food_waste_analysis_results\lowest_loss_crops.csv,Success
6,top_crops_measured,2 Crops,Crop ranking using measured data only,20,6,food_waste_analysis_results\top_crops_measured...,Success
7,product_group_summary,2 Crops,Loss by product group,12,7,food_waste_analysis_results\product_group_summ...,Success
8,crop_variability,2 Crops,Crops with the most uncertain loss,15,7,food_waste_analysis_results\crop_variability.csv,Success
9,modelled_vs_measured_gap,2 Crops,Modelled estimates versus measured data by crop,15,6,food_waste_analysis_results\modelled_vs_measur...,Success


## 5. Re-display a result (optional)

Every result is already displayed above. Change `RESULT_NAME` to revisit one from the `results` dictionary.

In [12]:
RESULT_NAME = "top_crops"

if RESULT_NAME not in results:
    print("Available result names:", list(results.keys()))
else:
    display(results[RESULT_NAME].head(20))

,crop_rank,commodity,product_group,records,countries,avg_loss_pct,median_loss_pct,p90_loss_pct
0,1,Green garlic,Vegetables,45,6,16.08,19.00,23.77
1,2,Mixed grain,Cereals,45,12,13.62,12.00,25.00
2,3,"Other fruits, n.e.c.",Fruits and nuts,82,16,12.94,10.58,24.90
3,4,Okra,Vegetables,91,5,11.79,10.30,25.00
4,5,Pineapples,Fruits and nuts,128,17,13.50,10.11,30.15
5,6,Eggplants (aubergines),Vegetables,87,11,12.63,10.00,21.70
6,6,Oranges,Fruits and nuts,93,16,13.08,10.00,33.49
7,6,Plantains and cooking bananas,Fruits and nuts,43,13,15.93,10.00,34.50
8,6,"Pumpkins, squash and gourds",Vegetables,62,9,11.34,10.00,22.85
9,6,Sweet potatoes,Roots and tubers,87,12,10.48,10.00,20.64


## 6. Close the database connection

Run this cell when you have finished.

In [13]:
if "connection" in globals() and connection and not connection.closed:
    connection.close()
    print("PostgreSQL connection closed.")

PostgreSQL connection closed.
